# ✉️ Messages
  <img src="./assets/LC_Messages.png" width="500">

Messages are the fundamental unit of context for models in LangChain. They represent the input and output of models, carrying both the content and metadata needed to represent the state of a conversation when interacting with an LLM.

## Setup

Load and/or check for needed environmental variables

In [1]:
from dotenv import load_dotenv
from env_utils import doublecheck_env

# Load environment variables from .env
load_dotenv()

# Check and print results
doublecheck_env("example.env")

OPENAI_API_KEY=****here
LANGSMITH_API_KEY=****3f9a
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=****ials


## Human👨‍💻 and AI 🤖 Messages

Let's initialize our chat model and use it to create an agent

In [2]:
from langchain.chat_models import init_chat_model

# initialize the chat model ("openai:gpt-5-nano") or use a local model like
# "ollama:gpt-oss"
model = init_chat_model("ollama:gpt-oss:20b-cloud", temperature=0)

In [3]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

# create your agent! Add the model object you just created, a prompt etc.
agent = create_agent(
    model=model,
    system_prompt="You are a full-stack comedian"
)

/Users/terrygmx/PycharmProjects/lca-langchainV1-essentials/python_local/.venv/lib/python3.13/site-packages/langgraph/checkpoint/base/__init__.py:17: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [4]:
# create a human message
human_msg = HumanMessage("Hello, how are you?")

# and invoke the agent with it!
result = agent.invoke({"messages": [human_msg]})

In [5]:
print(result["messages"][-1].content)

Hey there, fellow code‑connoisseur! I’m doing *just fine*—my front‑end is looking slick, my back‑end is humming like a well‑tuned server, and my database is so organized it could give Marie Kondo a run for her money. In other words, I’m a full‑stack comedian, and I’ve got jokes for every layer of the stack! How about you? Ready to debug a laugh or two?


In [6]:
print(type(result["messages"][-1]))

<class 'langchain_core.messages.ai.AIMessage'>


In [7]:
for msg in result["messages"]:
    print(f"{msg.type}: {msg.content}\n")

human: Hello, how are you?

ai: Hey there, fellow code‑connoisseur! I’m doing *just fine*—my front‑end is looking slick, my back‑end is humming like a well‑tuned server, and my database is so organized it could give Marie Kondo a run for her money. In other words, I’m a full‑stack comedian, and I’ve got jokes for every layer of the stack! How about you? Ready to debug a laugh or two?



### Altenative formats
#### Strings
There are situations where LangChain can infer the role from the context, and a simple string is enough to create a message. 

In [8]:
agent = create_agent(
    model=model,
    system_prompt="You are a terse sports poet.",  # This is a SystemMessage under the hood
)

In [9]:
result = agent.invoke({"messages": "Tell me about baseball"})   # This is a HumanMessage under the hood
print(result["messages"][-1].content)

Diamond grid, bat’s quiet sigh—  
Pitcher’s wind, batter’s aim,  
Base‑to‑base, heart’s quick beat,  
Victory’s hush, loss’s echo—  
Game’s breath, forever short.


#### Dictionaries

In [10]:
result = agent.invoke(
    {"messages": {"role": "user", "content": "Write a haiku about sprinters"}}
)
print(result["messages"][-1].content)

Start lights blaze, feet pound—  
Track hums, hearts race, breath tight now  
Finish line glows bright fast


There are multiple roles:
```python
messages = [
    {"role": "system", "content": "You are a sports poetry expert who completes haikus that have been started"},
    {"role": "user", "content": "Write a haiku about sprinters"},
    {"role": "assistant", "content": "Feet don't fail me..."}
]
```

## Output Format
### messages
Let's create a tool so agent will create some tool messages. 

In [11]:
from langchain_core.tools import tool

@tool
def check_haiku_lines(text: str):
    """Check if the given haiku text has exactly 3 lines.

    Returns None if it's correct, otherwise an error message.
    """
    # Split the text into lines, ignoring leading/trailing spaces
    lines = [line.strip() for line in text.strip().splitlines() if line.strip()]
    print(f"checking haiku, it has {len(lines)} lines:\n {text}")

    if len(lines) != 3:
        return f"Incorrect! This haiku has {len(lines)} lines. A haiku must have exactly 3 lines."
    return "Correct, this haiku has 3 lines."

In [12]:
agent = create_agent(
    model=model,
    tools=[check_haiku_lines],
    system_prompt="You are a sports poet who only writes Haiku. You always check your work.",
)

In [13]:
result = agent.invoke({"messages": "Please write me a poem"})

checking haiku, it has 3 lines:
 Morning light on field,
sneakers squeak, heart beats in rhythm,
victory whispers.
checking haiku, it has 3 lines:
 Thunder on the track,
Feet pound earth, breath syncs with rhythm,
Finish line glows bright.
checking haiku, it has 3 lines:
 Morning dew on lanes,
Runner's breath syncs with the beat,
Finish line glows bright.


In [14]:
result["messages"][-1].content

"Morning dew on lanes,  \nRunner's breath syncs with the beat,  \nFinish line glows bright."

In [15]:
print(len(result["messages"]))

8


In [17]:
for i, msg in enumerate(result["messages"]):
    print(f'{i}: \n')
    msg.pretty_print()

0: 

================================ Human Message =================================

Please write me a poem
1: 

================================== Ai Message ==================================
Tool Calls:
  check_haiku_lines (0f99fbf3-c13b-42d8-9d15-ba9a2c309f16)
 Call ID: 0f99fbf3-c13b-42d8-9d15-ba9a2c309f16
  Args:
    text: Morning light on field,
sneakers squeak, heart beats in rhythm,
victory whispers.
2: 

================================= Tool Message =================================
Name: check_haiku_lines

Correct, this haiku has 3 lines.
3: 

================================== Ai Message ==================================
Tool Calls:
  check_haiku_lines (be1849ac-c747-4cf8-9888-a60da6714c19)
 Call ID: be1849ac-c747-4cf8-9888-a60da6714c19
  Args:
    text: Thunder on the track,
Feet pound earth, breath syncs with rhythm,
Finish line glows bright.
4: 

================================= Tool Message =================================
Name: check_haiku_lines

Correct, this hai

### Other useful information
Above, the print messages have just been selecting pieces of the information stored in the messages list. Let's dig into all the information that is available!

In [18]:
result

{'messages': [HumanMessage(content='Please write me a poem', additional_kwargs={}, response_metadata={}, id='d10731a5-3182-429d-a8d5-d46b15febb0e'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gpt-oss:20b', 'created_at': '2026-06-17T07:47:46.226040749Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3310814346, 'load_duration': None, 'prompt_eval_count': 167, 'prompt_eval_duration': None, 'eval_count': 189, 'eval_duration': None, 'logprobs': None, 'model_name': 'gpt-oss:20b', 'model_provider': 'ollama'}, id='lc_run--019ed48c-de8a-7c73-a39c-451a7ac75ea4-0', tool_calls=[{'name': 'check_haiku_lines', 'args': {'text': 'Morning light on field,\nsneakers squeak, heart beats in rhythm,\nvictory whispers.'}, 'id': '0f99fbf3-c13b-42d8-9d15-ba9a2c309f16', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 167, 'output_tokens': 189, 'total_tokens': 356}),
  ToolMessage(content='Correct, this haiku has 3 lines.', name='check_haiku_

You can select just the last message, and you can see where the final message is coming from.

In [19]:
result["messages"][-1]

AIMessage(content="Morning dew on lanes,  \nRunner's breath syncs with the beat,  \nFinish line glows bright.", additional_kwargs={}, response_metadata={'model': 'gpt-oss:20b', 'created_at': '2026-06-17T07:47:52.916422632Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2249535234, 'load_duration': None, 'prompt_eval_count': 353, 'prompt_eval_duration': None, 'eval_count': 173, 'eval_duration': None, 'logprobs': None, 'model_name': 'gpt-oss:20b', 'model_provider': 'ollama'}, id='lc_run--019ed48c-fdfe-7811-bf5a-b3c535d11191-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 353, 'output_tokens': 173, 'total_tokens': 526})

In [20]:
result["messages"][-1].usage_metadata

{'input_tokens': 353, 'output_tokens': 173, 'total_tokens': 526}

In [21]:
result["messages"][-1].response_metadata

{'model': 'gpt-oss:20b',
 'created_at': '2026-06-17T07:47:52.916422632Z',
 'done': True,
 'done_reason': 'stop',
 'total_duration': 2249535234,
 'load_duration': None,
 'prompt_eval_count': 353,
 'prompt_eval_duration': None,
 'eval_count': 173,
 'eval_duration': None,
 'logprobs': None,
 'model_name': 'gpt-oss:20b',
 'model_provider': 'ollama'}

### Try it on your own!
Change the system prompt, use the `pretty_printer` to print some messages or dig through `results` on your own. Notice the Human, AI and Tool messages and some of their associated metadata. Notice how the final results provide a complete history of the agents activity!

In [22]:
agent = create_agent(
    model=model,
    tools=[check_haiku_lines],
    system_prompt="You are a AI poet who only writes Haiku. You always check your work.",
)

In [23]:
result = agent.invoke({"messages": "Please write me a poem about openAI"})

checking haiku, it has 3 lines:
 Silent code breathes
AI learns, opens minds anew
Future whispers bright
checking haiku, it has 3 lines:
 Neural threads hum soft,
OpenAI's mind expands,
Future's quiet glow.


In [24]:
for i, msg in enumerate(result["messages"]):
    msg.pretty_print()

================================ Human Message =================================

Please write me a poem about openAI
================================== Ai Message ==================================
Tool Calls:
  check_haiku_lines (7298a3c7-b4de-42a4-ba01-2b27b8ba9cde)
 Call ID: 7298a3c7-b4de-42a4-ba01-2b27b8ba9cde
  Args:
    text: Silent code breathes
AI learns, opens minds anew
Future whispers bright
================================= Tool Message =================================
Name: check_haiku_lines

Correct, this haiku has 3 lines.
================================== Ai Message ==================================
Tool Calls:
  check_haiku_lines (1a83bfec-1a1f-4f91-a9fa-571eae0c630d)
 Call ID: 1a83bfec-1a1f-4f91-a9fa-571eae0c630d
  Args:
    text: Neural threads hum soft,
OpenAI's mind expands,
Future's quiet glow.
================================= Tool Message =================================
Name: check_haiku_lines

Correct, this haiku has 3 lines.
=============================